# Fine-tune Llama 3.1 8B Instruct for IT-support ticket routing (Unsloth + LoRA)

**Runtime required:** `Runtime -> Change runtime type -> T4 GPU` (free tier is enough).

What this notebook does:
1. Installs Unsloth.
2. Loads `unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit` (4-bit, fits in a T4).
3. Trains a LoRA adapter on `NetworkTeam.json`, `SecurityTeam.json`, `SoftwareTeam.json` (1,500 rows).
4. Evaluates routing accuracy on `test_set.json` (100 held-out rows the model never saw).
5. Exports a **Q4_K_M GGUF** you can load into Ollama on your Mac, and saves it to Google Drive.

Run the cells top to bottom. Total time on a free T4 is roughly 45-60 minutes (training ~15 min, GGUF export ~20 min).

In [ ]:
%%capture
# If this cell ever breaks, copy the install cell from the official Unsloth Llama 3.1 Colab notebook.
!pip install unsloth
!pip install --upgrade --no-cache-dir --no-deps "unsloth_zoo" "git+https://github.com/unslothai/unsloth.git"


## 1. Upload the dataset files
Upload **NetworkTeam.json, SecurityTeam.json, SoftwareTeam.json, test_set.json** from your `ITSupportTeam` folder when the file picker appears.

In [ ]:
from google.colab import files
import os

uploaded = files.upload()
for name in ["NetworkTeam.json", "SecurityTeam.json", "SoftwareTeam.json", "test_set.json"]:
    assert os.path.exists(name), f"Missing {name} - please upload it"
print("All dataset files present.")


## 2. Load the base model (Llama 3.1 8B Instruct, 4-bit)

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 1024
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = None,          # auto: float16 on T4
    load_in_4bit = True,
)


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)


## 3. Build the training dataset
Each JSON row becomes a 3-turn chat: `system` = instruction, `user` = the issue, `assistant` = `Team: ... | Assigned to ...: steps`.
We use the Llama 3.1 chat template so the GGUF behaves correctly in Ollama.

In [ ]:
import json
from datasets import Dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template = "llama-3.1")

rows = []
for f in ["NetworkTeam.json", "SecurityTeam.json", "SoftwareTeam.json"]:
    rows += json.load(open(f))
print("training rows:", len(rows))

SYSTEM_PROMPT = rows[0]["instruction"]
print("system prompt:", SYSTEM_PROMPT)

def to_text(r):
    msgs = [
        {"role": "system",    "content": r["instruction"]},
        {"role": "user",      "content": r["input"]},
        {"role": "assistant", "content": r["output"]},
    ]
    return {"text": tokenizer.apply_chat_template(msgs, tokenize = False, add_generation_prompt = False)}

dataset = Dataset.from_list(rows).shuffle(seed = 3407).map(to_text)
print(dataset[0]["text"])


## 4. Train (LoRA, 3 epochs, loss computed only on the assistant answer)

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length = max_seq_length,
        packing = False,
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 4,   # effective batch 16
        warmup_steps = 10,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

# Only learn to produce the assistant turn, not to repeat the system/user text.
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|start_header_id|>user<|end_header_id|>\n\n",
    response_part    = "<|start_header_id|>assistant<|end_header_id|>\n\n",
)


In [ ]:
# Sanity check: the masked part shows only the assistant answer is trained on.
space = tokenizer(" ", add_special_tokens = False).input_ids[0]
print(tokenizer.decode([space if x == -100 else x for x in trainer.train_dataset[0]["labels"]]))


In [ ]:
trainer_stats = trainer.train()
print(f"Training took {trainer_stats.metrics['train_runtime']/60:.1f} minutes")


## 5. Quick manual test

In [ ]:
FastLanguageModel.for_inference(model)

def route(issue, max_new_tokens = 96):
    msgs = [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": issue}]
    inputs = tokenizer.apply_chat_template(msgs, tokenize = True,
                                           add_generation_prompt = True,
                                           return_tensors = "pt").to("cuda")
    out = model.generate(input_ids = inputs, max_new_tokens = max_new_tokens,
                         do_sample = False, pad_token_id = tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens = True).strip()

for q in [
    "Wi-Fi keeps disconnecting in the 2nd floor meeting rooms.",
    "Who should handle this issue: an employee got an email asking them to re-enter their Office 365 password?",
    "Jenkins deploy stage has been failing since this morning.",
    "I need a Visio license for the new architect.",
    "Someone needs access to the finance shared drive.",
    "The wall jack at my desk has no link light.",
]:
    print("Q:", q)
    print("A:", route(q))
    print()


## 6. Evaluate on the 100 held-out test tickets

In [ ]:
import re

test = json.load(open("test_set.json"))
emp_re = re.compile(r"Assigned to ([^:]+):")
team_re = re.compile(r"Team:\s*([A-Za-z]+)")

correct_emp = correct_team = 0
per_emp = {}
mistakes = []
for r in test:
    pred = route(r["input"], max_new_tokens = 64)
    want_emp  = emp_re.search(r["output"]).group(1).strip()
    want_team = team_re.search(r["output"]).group(1).strip()
    got_emp   = (emp_re.search(pred) or [None, "?"])[1].strip()
    got_team  = (team_re.search(pred) or [None, "?"])[1].strip()
    per_emp.setdefault(want_emp, [0, 0])
    per_emp[want_emp][1] += 1
    if got_emp == want_emp:
        correct_emp += 1
        per_emp[want_emp][0] += 1
    else:
        mistakes.append((r["input"], want_emp, got_emp))
    if got_team == want_team:
        correct_team += 1

print(f"Team accuracy:     {correct_team}/{len(test)} = {100*correct_team/len(test):.1f}%")
print(f"Employee accuracy: {correct_emp}/{len(test)} = {100*correct_emp/len(test):.1f}%")
for e, (c, n) in sorted(per_emp.items()):
    print(f"  {e:16s} {c}/{n}")
print("\nMistakes:")
for q, want, got in mistakes:
    print(f"- {q}\n    expected {want}, got {got}")


## 7. Save the LoRA adapter (small, ~160 MB) as a backup

In [ ]:
model.save_pretrained("lora_model")
tokenizer.save_pretrained("lora_model")
!zip -qr lora_model.zip lora_model && ls -lh lora_model.zip


## 8. Export a GGUF for Ollama (Q4_K_M, ~4.9 GB)
This merges the LoRA into the base weights and quantizes. It takes ~15-25 minutes and compiles llama.cpp the first time.
If the free runtime runs out of RAM here, restart with a High-RAM runtime, or skip to the adapter fallback described in the README.

In [ ]:
model.save_pretrained_gguf("llama31-itsupport-gguf", tokenizer, quantization_method = "q4_k_m")
!ls -lh llama31-itsupport-gguf


## 9. Copy the GGUF to Google Drive
Downloading 4.9 GB straight from Colab is slow and often fails; Drive is reliable. After this cell finishes, download the file from
`My Drive/llama31-itsupport/` with the Google Drive desktop app or the browser.

In [ ]:
from google.colab import drive
import glob, shutil, os

drive.mount("/content/drive")
dest = "/content/drive/MyDrive/llama31-itsupport"
os.makedirs(dest, exist_ok = True)

# Unsloth may write to "llama31-itsupport-gguf/" or "llama31-itsupport-gguf_gguf/" and also leaves a 16 GB F16 file
# behind, so search for the Q4_K_M file specifically.
ggufs = [p for p in glob.glob("**/*.gguf", recursive = True) if "q4_k_m" in p.lower()]
assert ggufs, "No Q4_K_M .gguf found - check the previous cell's output"
src = ggufs[0]
print("Found:", src)
shutil.copy(src, f"{dest}/llama31-itsupport.Q4_K_M.gguf")
shutil.copy("lora_model.zip", f"{dest}/lora_model.zip")
print("Copied to Drive:")
!ls -lh "$dest"


## 10. Next steps on your Mac
1. Download `llama31-itsupport.Q4_K_M.gguf` from Google Drive into your `ITSupportTeam` folder.
2. `ollama create it-support -f Modelfile.llama31-itsupport`
3. `ollama run it-support "Wi-Fi keeps dropping in the lobby"`
4. `python3 test_routing.py` to score the model on `test_set.json` locally.

(Optional) Push to Hugging Face instead of Drive:
```python
model.push_to_hub_gguf("YOUR_HF_USERNAME/llama31-itsupport", tokenizer, quantization_method="q4_k_m", token="hf_...")
```
Then locally: `ollama run hf.co/YOUR_HF_USERNAME/llama31-itsupport:Q4_K_M`